# K-Fold Cross-Validation and Hold-Out Validation

In [ ]:
# =============================================================================
# CELL 1: Import Libraries
# This cell imports all necessary Python libraries for machine learning
# =============================================================================

import numpy as np                      # NumPy: For numerical operations and array handling
import matplotlib.pyplot as plt         # Matplotlib: For creating visualizations and plots

# Sklearn (Scikit-learn) imports:
from sklearn.datasets import load_iris  # Load the famous Iris flower dataset
from sklearn.model_selection import train_test_split, KFold, cross_val_score  
# train_test_split: Splits data into train/test sets
# KFold: Creates K equal-sized folds for cross-validation
# cross_val_score: Automatically performs cross-validation and returns scores

from sklearn.linear_model import LogisticRegression    # Logistic Regression classifier
from sklearn.tree import DecisionTreeClassifier        # Decision Tree classifier
from sklearn.neighbors import KNeighborsClassifier     # K-Nearest Neighbors classifier
from sklearn.svm import SVC                            # Support Vector Machine classifier

from sklearn.metrics import accuracy_score, classification_report  
# accuracy_score: Calculates prediction accuracy (correct/total)
# classification_report: Detailed metrics (precision, recall, f1-score)

from sklearn.base import clone          # Creates a copy of a model with same parameters

import warnings                         # Python warnings module
warnings.filterwarnings('ignore')       # Suppress warning messages for cleaner output

print('Libraries imported!')            # Confirmation message

In [ ]:
# =============================================================================
# CELL 2: Load the Iris Dataset
# This cell loads the Iris dataset - a classic ML dataset with 150 flower samples
# Each flower has 4 measurements and belongs to 1 of 3 species
# =============================================================================

iris = load_iris()      # Load the Iris dataset (returns a Bunch object - like a dictionary)
                        # Contains: data, target, feature_names, target_names, DESCR

X = iris.data           # X = Feature matrix (input data)
                        # Shape: (150, 4) - 150 samples, 4 features per sample
                        # Features: sepal length, sepal width, petal length, petal width

y = iris.target         # y = Target vector (labels/output)
                        # Shape: (150,) - 150 labels
                        # Values: 0 = setosa, 1 = versicolor, 2 = virginica

# Print dataset information:
print(f'Dataset shape: {X.shape}')              # Shows (150, 4) - rows x columns
print(f'Number of classes: {len(np.unique(y))}') # Shows 3 - three flower species
print(f'Class distribution: {np.bincount(y)}')   # Shows [50, 50, 50] - balanced classes

In [ ]:
# =============================================================================
# CELL 3: Trace Variables - Detailed Exploration of Iris Dataset
# This cell helps you understand WHAT each variable contains
# Run this cell to see the internal structure of iris, X, and y
# =============================================================================

print('='*60)
print('VARIABLE TRACE: Iris Dataset')
print('='*60)

# --- 1. Explore the 'iris' object ---
# iris is a "Bunch" object (like a dictionary with dot notation)
print('\n1. iris object type:', type(iris))       # Shows: sklearn.utils._bunch.Bunch
print('   Available keys:', list(iris.keys()))   # Shows all accessible attributes

# --- 2. Feature names - What do the 4 columns in X represent? ---
print('\n2. Feature names (columns of X):')
for i, name in enumerate(iris.feature_names):    # enumerate gives index + value
    print(f'   Column {i}: {name}')              # Prints: sepal length, sepal width, etc.

# --- 3. Target names - What do the numbers 0, 1, 2 mean? ---
print('\n3. Target names (class labels):')
for i, name in enumerate(iris.target_names):     # Loop through class names
    print(f'   {i} = {name}')                    # 0=setosa, 1=versicolor, 2=virginica

# --- 4. X variable - The feature matrix (input data) ---
print('\n4. X (Feature Data):')
print(f'   Type: {type(X)}')                     # numpy.ndarray (N-dimensional array)
print(f'   Shape: {X.shape} (150 samples, 4 features)')  # (rows, columns)
print(f'   First 5 rows:')                       # Preview of the data
print(X[:5])                                     # X[:5] means first 5 rows (slicing)

# --- 5. y variable - The target vector (labels) ---
print('\n5. y (Target Labels):')
print(f'   Type: {type(y)}')                     # numpy.ndarray
print(f'   Shape: {y.shape}')                    # (150,) - 1D array with 150 elements
print(f'   Unique values: {np.unique(y)}')       # [0, 1, 2] - three unique classes
print(f'   Full array: {y}')                     # All 150 labels (50 zeros, 50 ones, 50 twos)

# --- 6. Visual summary - Sample data with human-readable labels ---
print('\n6. Sample Data with Labels:')
print(f'   {"Sample":<8} {"Sepal L":<10} {"Sepal W":<10} {"Petal L":<10} {"Petal W":<10} {"Class":<10}')
print('-'*60)
for i in [0, 50, 100]:                           # Index 0, 50, 100 = one from each class
    # X[i,0] = row i, column 0 (sepal length)
    # iris.target_names[y[i]] converts number to flower name
    print(f'   {i:<8} {X[i,0]:<10.1f} {X[i,1]:<10.1f} {X[i,2]:<10.1f} {X[i,3]:<10.1f} {iris.target_names[y[i]]:<10}')

## Part 1: Hold-Out Validation

In [ ]:
# =============================================================================
# CELL 4: Hold-Out Split Function
# PURPOSE: Split data into 3 parts: Training (60%), Validation (20%), Test (20%)
# 
# WHY 3 SPLITS?
# - Training set: Used to train the model (model learns from this)
# - Validation set: Used to tune hyperparameters (adjust model settings)
# - Test set: Used for final evaluation (never seen during training)
# =============================================================================

def hold_out_split(X, y, train_ratio=0.6, val_ratio=0.2, test_ratio=0.2, random_state=42):
    """
    Splits data into training, validation, and test sets.
    
    Parameters:
    - X: Feature matrix (input data)
    - y: Target vector (labels)
    - train_ratio: Proportion for training (default 60%)
    - val_ratio: Proportion for validation (default 20%)
    - test_ratio: Proportion for testing (default 20%)
    - random_state: Seed for reproducibility (same split every time)
    """
    
    # STEP 1: First split - Separate the TEST set from the rest
    # test_size=0.2 means 20% goes to test, 80% stays in X_temp
    # stratify=y ensures each split has same class proportions as original
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y,                           # Input data and labels
        test_size=test_ratio,           # 20% for test
        random_state=random_state,      # Reproducible results
        stratify=y                      # Keep class balance
    )
    
    # STEP 2: Second split - Separate TRAIN and VALIDATION from X_temp
    # We need to recalculate the ratio because we're splitting from 80%, not 100%
    # val_ratio_adjusted = 0.2 / (0.6 + 0.2) = 0.2 / 0.8 = 0.25 (25% of remaining)
    val_ratio_adjusted = val_ratio / (train_ratio + val_ratio)
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,                 # The 80% remaining data
        test_size=val_ratio_adjusted,   # 25% of 80% = 20% of original
        random_state=random_state,      # Same seed for reproducibility
        stratify=y_temp                 # Keep class balance
    )
    
    # Return all 6 arrays: features and labels for train, val, and test
    return X_train, X_val, X_test, y_train, y_val, y_test

# --- Apply the hold-out split to our Iris data ---
X_train, X_val, X_test, y_train, y_val, y_test = hold_out_split(X, y)

# --- Print the results to verify the split ---
print('Hold-Out Split Results:')
print(f'Training set:   {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)')   # 90 samples (60%)
print(f'Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)')       # 30 samples (20%)
print(f'Test set:       {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)')     # 30 samples (20%)

In [ ]:
# =============================================================================
# CELL 5: Training with Hold-Out Validation
# PURPOSE: Train a model and evaluate it on train/validation/test sets
# 
# WORKFLOW:
# 1. Create a Logistic Regression model
# 2. Train (fit) the model on training data
# 3. Evaluate accuracy on all three sets
# =============================================================================

print('='*50)
print('HOLD-OUT VALIDATION: Training & Evaluation')
print('='*50)

# --- Step 1: Create the model ---
# LogisticRegression: A classification algorithm for predicting categories
# max_iter=200: Maximum iterations for the optimization algorithm to converge
model_holdout = LogisticRegression(max_iter=200)

# --- Step 2: Train (fit) the model ---
# .fit() learns patterns from training data (X_train) and labels (y_train)
# The model adjusts its internal weights to minimize prediction errors
model_holdout.fit(X_train, y_train)

# --- Step 3: Evaluate on all three sets ---

# Training Accuracy: How well does model predict data it was trained on?
# .predict(X_train) generates predictions, accuracy_score compares with actual y_train
train_accuracy = accuracy_score(y_train, model_holdout.predict(X_train))

# Validation Accuracy: How well does model predict unseen validation data?
# This helps detect overfitting (model memorizing instead of learning)
val_accuracy = accuracy_score(y_val, model_holdout.predict(X_val))

# Test Accuracy: Final evaluation on completely unseen test data
# This is the "real" performance estimate
y_test_pred = model_holdout.predict(X_test)      # Get predictions for test set
test_accuracy = accuracy_score(y_test, y_test_pred)  # Compare with actual labels

# --- Print results ---
print(f'1. Training Accuracy: {train_accuracy:.4f}')    # Expected: High (model saw this data)
print(f'2. Validation Accuracy: {val_accuracy:.4f}')    # Expected: Slightly lower
print(f'3. Test Accuracy: {test_accuracy:.4f}')         # Final performance metric

## Part 2: K-Fold Cross-Validation

In [ ]:
# =============================================================================
# CELL 6: K-Fold Cross-Validation Function
# PURPOSE: Implement K-Fold CV from scratch to understand how it works
# 
# WHAT IS K-FOLD?
# - Divide data into K equal parts (folds)
# - Train K times, each time using a different fold as validation
# - Average the results for a more reliable performance estimate
# 
# ADVANTAGE: Every data point is used for both training and validation
# =============================================================================

def k_fold_cross_validation(X, y, model, k=5, random_state=42):
    """
    Performs K-Fold Cross-Validation manually.
    
    Parameters:
    - X: Feature matrix
    - y: Target vector
    - model: The ML model to evaluate
    - k: Number of folds (default 5)
    - random_state: Seed for reproducibility
    
    Returns:
    - Dictionary with fold scores, mean, and standard deviation
    """
    
    # Create KFold object
    # n_splits=k: Divide data into k equal parts
    # shuffle=True: Randomly shuffle data before splitting
    # random_state: Ensures same splits every time we run
    kfold = KFold(n_splits=k, shuffle=True, random_state=random_state)
    
    fold_scores = []  # List to store validation accuracy for each fold
    
    print(f'K-Fold Cross-Validation (K={k})')
    print('-' * 40)
    
    # --- Loop through each fold ---
    # kfold.split(X) generates indices for train/validation splits
    # enumerate(..., 1) starts counting from 1 instead of 0
    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X), 1):
        
        # Get training data for this fold using the indices
        X_train_fold = X[train_idx]      # Training features (80% of data)
        y_train_fold = y[train_idx]      # Training labels
        
        # Get validation data for this fold
        X_val_fold = X[val_idx]          # Validation features (20% of data)
        y_val_fold = y[val_idx]          # Validation labels
        
        # Clone the model (create fresh copy with same parameters)
        # This ensures each fold starts with an untrained model
        model_fold = clone(model)
        
        # Train the model on this fold's training data
        model_fold.fit(X_train_fold, y_train_fold)
        
        # Calculate training accuracy for this fold
        train_score = accuracy_score(y_train_fold, model_fold.predict(X_train_fold))
        
        # Calculate validation accuracy for this fold
        val_score = accuracy_score(y_val_fold, model_fold.predict(X_val_fold))
        
        # Store the validation score
        fold_scores.append(val_score)
        
        # Print results for this fold
        print(f'Fold {fold_idx}: Train Acc = {train_score:.4f}, Val Acc = {val_score:.4f}')
    
    # --- Calculate summary statistics ---
    mean_score = np.mean(fold_scores)    # Average of all fold scores
    std_score = np.std(fold_scores)      # Standard deviation (measures variance)
    
    print('-' * 40)
    print(f'Mean Accuracy: {mean_score:.4f} (+/- {std_score:.4f})')
    
    # Return results as a dictionary
    return {'fold_scores': fold_scores, 'mean': mean_score, 'std': std_score}

In [ ]:
# =============================================================================
# CELL 7: Run K-Fold Cross-Validation
# PURPOSE: Execute our custom K-Fold function on the Iris dataset
# 
# With K=5:
# - Data is split into 5 equal parts (30 samples each)
# - Model trains 5 times
# - Each time, 120 samples for training, 30 for validation
# =============================================================================

print('='*50)
print('K-FOLD CROSS-VALIDATION: Training & Evaluation')
print('='*50)

# Create a Logistic Regression model (same as before)
model_kfold = LogisticRegression(max_iter=200)

# Run our custom K-Fold cross-validation function
# k=5 means 5-fold cross-validation (most common choice)
results = k_fold_cross_validation(X, y, model_kfold, k=5)

# 'results' contains:
# - results['fold_scores']: List of 5 accuracy values (one per fold)
# - results['mean']: Average accuracy across all folds
# - results['std']: Standard deviation (how much scores vary)

In [ ]:
# =============================================================================
# CELL 8: Using sklearn's Built-in cross_val_score
# PURPOSE: Compare our manual implementation with sklearn's version
# 
# cross_val_score does the same thing as our function but in one line!
# It's the recommended way to do K-Fold CV in practice
# =============================================================================

print('Using sklearn cross_val_score:')

# cross_val_score parameters:
# - LogisticRegression(max_iter=200): The model to evaluate
# - X: Feature matrix (all data)
# - y: Target vector (all labels)
# - cv=5: Number of folds (5-fold cross-validation)
cv_scores = cross_val_score(LogisticRegression(max_iter=200), X, y, cv=5)

# cv_scores is a numpy array with 5 accuracy values (one per fold)
print(f'Fold scores: {cv_scores}')                          # Array of 5 scores
print(f'Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')  # Summary

# NOTE: Results may differ slightly from our manual implementation
# because sklearn uses different random shuffling by default

## Part 3: Visualization

In [ ]:
# =============================================================================
# CELL 9: Visual Comparison - Bar Charts
# PURPOSE: Create side-by-side visualizations comparing Hold-Out vs K-Fold
# 
# Plot 1 (Left): Hold-Out results showing train/val/test accuracy
# Plot 2 (Right): K-Fold results showing accuracy for each of 5 folds
# =============================================================================

# Create a figure with 2 subplots side by side
# figsize=(14, 5): 14 inches wide, 5 inches tall
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ======================== PLOT 1: Hold-Out Results ========================
ax1 = axes[0]                                    # Get the first (left) subplot

# Data for the bar chart
sets = ['Training', 'Validation', 'Test']        # X-axis labels
holdout_scores = [train_accuracy, val_accuracy, test_accuracy]  # Y-axis values
colors = ['#2ecc71', '#3498db', '#e74c3c']       # Green, Blue, Red (hex colors)

# Create bar chart
# ax1.bar(x_labels, heights, color, edgecolor)
bars1 = ax1.bar(sets, holdout_scores, color=colors, edgecolor='black')

ax1.set_ylim(0, 1.1)                             # Y-axis range: 0 to 1.1 (for labels)
ax1.set_ylabel('Accuracy')                       # Y-axis label
ax1.set_title('Hold-Out Validation Results')    # Plot title

# Add value labels on top of each bar
for bar, score in zip(bars1, holdout_scores):    # Loop through bars and scores
    # bar.get_x(): Left edge of bar
    # bar.get_width()/2: Half the bar width (centers the text)
    # bar.get_height() + 0.02: Position slightly above the bar
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{score:.3f}', ha='center')        # ha='center' centers horizontally

# ======================== PLOT 2: K-Fold Results ========================
ax2 = axes[1]                                    # Get the second (right) subplot

# Data for the bar chart
folds = [f'Fold {i+1}' for i in range(5)]        # ['Fold 1', 'Fold 2', ..., 'Fold 5']

# Create bar chart for fold scores
bars2 = ax2.bar(folds, results['fold_scores'], color='#9b59b6', edgecolor='black')

# Add horizontal line showing mean accuracy
# linestyle='--': Dashed line
ax2.axhline(y=results['mean'], color='red', linestyle='--', 
            label=f"Mean = {results['mean']:.3f}")

ax2.set_ylim(0, 1.1)                             # Y-axis range
ax2.set_ylabel('Accuracy')                       # Y-axis label
ax2.set_title('5-Fold Cross-Validation Results') # Plot title
ax2.legend()                                     # Show legend (for the mean line)

# Add value labels on top of each bar
for bar, score in zip(bars2, results['fold_scores']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{score:.3f}', ha='center')

# Adjust layout to prevent overlap
plt.tight_layout()

# Display the plots
plt.show()

In [ ]:
# =============================================================================
# CELL 10: K-Fold Visualization Diagram
# PURPOSE: Visual explanation of how K-Fold splits data across iterations
# 
# This diagram shows:
# - 5 columns = 5 data folds
# - 5 rows = 5 iterations (training rounds)
# - Green = Training data, Red = Validation data
# - Each iteration uses a different fold for validation
# =============================================================================

from matplotlib.patches import Patch            # For creating custom legend

# Create figure and axes
fig, ax = plt.subplots(figsize=(12, 4))         # 12 inches wide, 4 inches tall
k = 5                                            # Number of folds

# --- Draw the grid of rectangles ---
for fold in range(k):                            # For each iteration (row)
    for i in range(k):                           # For each fold (column)
        # If column index equals fold index, this is the validation fold (red)
        # Otherwise, it's a training fold (green)
        color = '#e74c3c' if i == fold else '#2ecc71'
        
        # Create a rectangle at position (i, k-1-fold)
        # (i, k-1-fold) places it at column i, row from top
        # 0.9, 0.8 = width and height (slightly smaller than 1 for spacing)
        rect = plt.Rectangle((i, k-1-fold), 0.9, 0.8, 
                              color=color, edgecolor='black')
        ax.add_patch(rect)                       # Add rectangle to plot

# --- Configure the axes ---
ax.set_xlim(-0.5, k+1)                          # X-axis limits
ax.set_ylim(-0.5, k+0.5)                        # Y-axis limits
ax.set_xlabel('Data Folds')                     # X-axis label
ax.set_ylabel('Iteration')                      # Y-axis label
ax.set_title('K-Fold Cross-Validation Visualization (K=5)')  # Title

# Set tick positions and labels for X-axis (centered on rectangles)
ax.set_xticks([i+0.45 for i in range(k)])       # Tick positions
ax.set_xticklabels([f'Fold {i+1}' for i in range(k)])  # Tick labels

# Set tick positions and labels for Y-axis
ax.set_yticks([i+0.4 for i in range(k)])        # Tick positions
ax.set_yticklabels([f'Iter {k-i}' for i in range(k)])  # Iter 5, 4, 3, 2, 1 from top

# --- Create legend ---
# Patch creates a colored square for the legend
legend_elements = [
    Patch(facecolor='#2ecc71', edgecolor='black', label='Training'),
    Patch(facecolor='#e74c3c', edgecolor='black', label='Validation')
]
ax.legend(handles=legend_elements, loc='upper right')  # Place legend in upper right

plt.tight_layout()                              # Adjust layout
plt.show()                                      # Display the plot

## Part 4: Summary Comparison

In [ ]:
# =============================================================================
# CELL 11: Summary Comparison Table
# PURPOSE: Print a formatted table comparing Hold-Out vs K-Fold methods
# 
# This is a TEXT-BASED summary (not a visualization)
# Helps understand when to use each validation method
# =============================================================================

print('='*70)
print('COMPARISON: Hold-Out vs K-Fold Cross-Validation')
print('='*70)
print()

# Print table header
# {text:<25} means left-align text in a 25-character wide field
print(f'{"Aspect":<25} {"Hold-Out":<25} {"K-Fold CV":<25}')
print('-'*70)

# Print comparison rows
# Data Usage: Hold-Out uses data once; K-Fold uses all data K times
print(f'{"Data Usage":<25} {"Single split":<25} {"All data used K times":<25}')

# Training Iterations: Hold-Out trains once; K-Fold trains K times
print(f'{"Training Iterations":<25} {"1":<25} {"K":<25}')

# Variance: Hold-Out has higher variance (depends on random split)
print(f'{"Variance in Estimate":<25} {"Higher":<25} {"Lower":<25}')

# Computational Cost: K-Fold is K times more expensive
print(f'{"Computational Cost":<25} {"Lower":<25} {"Higher (K times)":<25}')

# Best Use Case: Hold-Out for big data; K-Fold for smaller datasets
print(f'{"Best For":<25} {"Large datasets":<25} {"Small/medium datasets":<25}')

print('='*70)

## Part 5: Model Comparison

In [ ]:
# =============================================================================
# CELL 12: Model Comparison
# PURPOSE: Compare multiple ML models using both Hold-Out and K-Fold validation
# 
# This demonstrates that validation methods can be applied to ANY model
# We compare: Logistic Regression, Decision Tree, KNN, and SVM
# =============================================================================

# --- Define a dictionary of models to compare ---
# Key = model name (string), Value = model object
models = {
    'Logistic Regression': LogisticRegression(max_iter=200),  # Linear classifier
    'Decision Tree': DecisionTreeClassifier(max_depth=5),      # Tree-based classifier
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),          # Instance-based classifier
    'SVM': SVC(kernel='rbf')                                    # Support Vector Machine
}

print('='*70)
print('MODEL COMPARISON')
print('='*70)

# Print table header
print(f'{"Model":<25} {"Hold-Out (Test)":<20} {"5-Fold CV (Mean +/- Std)":<25}')
print('-'*70)

# --- Loop through each model and evaluate ---
for name, model in models.items():
    
    # ===== Hold-Out Evaluation =====
    # Train on training set (same split we created earlier)
    model.fit(X_train, y_train)
    
    # Predict on test set and calculate accuracy
    holdout_score = accuracy_score(y_test, model.predict(X_test))
    
    # ===== K-Fold Cross-Validation =====
    # Use sklearn's cross_val_score for convenience
    # This uses ALL data (X, y), not just the hold-out splits
    cv_scores = cross_val_score(model, X, y, cv=5)
    
    # Print results for this model
    # holdout_score: Single test accuracy
    # cv_scores.mean(): Average of 5 fold accuracies
    # cv_scores.std(): Standard deviation of fold accuracies
    print(f'{name:<25} {holdout_score:<20.4f} {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

# INTERPRETATION:
# - Models with similar Hold-Out and K-Fold scores are stable
# - Large difference suggests Hold-Out result may be lucky/unlucky
# - K-Fold (+/- std) shows how much performance varies

## Questions You Can Answer

1. **What is Hold-Out validation?** - Splitting data into train/validation/test sets
2. **What is K-Fold Cross-Validation?** - Dividing data into K folds, training K times
3. **Why K-Fold is better?** - Uses all data, lower variance, more reliable estimate